# Verificação de Idempotência — Camada Gold

**Tech Challenge Fase 2 — POSTECH AI Scientist**

Valida que o script de preparação da camada **Gold** pode ser executado múltiplas vezes sem alterar o estado final do data lake — propriedade essencial para reprocessos e retries seguros em produção.

---

**Por que idempotência importa?** Numa pipeline orquestrada (Step Functions, retries automáticos, backfills manuais), qualquer job pode ser executado mais de uma vez sobre o mesmo insumo — por falha transitória, reprocesso ou operação humana. Um job **idempotente** produz exatamente o mesmo estado final do lake em todas as execuções: sem duplicar registros, sem apagar histórico, sem variar conteúdo.

**Como o job Glue garante isso?** Escrita Parquet com `mode("overwrite")` + `partitionBy("ano")` + `spark.sql.sources.partitionOverwriteMode=dynamic`: cada execução sobrescreve **apenas as partições `ano=YYYY` presentes no lote**, deixando as demais intactas.

**Como este notebook verifica?** Usamos a simulação local fiel da pipeline (`pipeline_local.py`, pandas), executamos a camada **duas vezes** e comparamos *snapshots* do lake — uma linha por partição física com contagem de linhas e hash MD5 do conteúdo de negócio (colunas voláteis de timestamp excluídas, pois mudam legitimamente a cada execução). Idempotência confirmada = mesmas partições, mesmas contagens, mesmos hashes.

In [1]:
# Simulação local fiel dos jobs Glue (mesma lógica de transformação,
# DQ e particionamento Hive-style por ano) — ver notebooks/pipeline_local.py
import tempfile
from pathlib import Path

import pandas as pd

from pipeline_local import (
    run_bronze, run_silver, run_gold,
    snapshot_camada, comparar_snapshots,
)

DADOS_DIR = Path('../dados')
LAKE_DIR = Path(tempfile.mkdtemp(prefix='lake_'))
print(f'Data lake local desta verificação: {LAKE_DIR}')

Data lake local desta verificação: /sessions/gallant-youthful-faraday/tmp/lake_ungxkrdo


## Preparação das camadas anteriores

A verificação isola a camada em teste: as camadas a montante são materializadas **uma única vez** e permanecem constantes durante o experimento.

In [2]:
# Bronze e Silver materializadas uma única vez — insumo constante
run_bronze(DADOS_DIR, LAKE_DIR)
run_silver(LAKE_DIR)

{'indicador_municipio': (23995, 0),
 'indicador_uf': (145, 0),
 'meta_brasil': (3, 0),
 'meta_uf': (54, 0),
 'meta_municipio': (10704, 0)}

## Execução 1 — estado inicial da camada Gold

In [3]:
resultado_exec1 = run_gold(LAKE_DIR)
print('Registros processados na execução 1:')
resultado_exec1

Registros processados na execução 1:


{'alfabetizacao_por_municipio': 10896,
 'evolucao_temporal': 49,
 'ranking_municipios': 10896,
 'comparacao_metas_nacionais': 1}

In [4]:
def listar_particoes(base):
    """Lista a estrutura física de partições gravada no lake local."""
    for p in sorted((LAKE_DIR / base).rglob('ano=*')):
        n = len(pd.read_parquet(p / 'dados.parquet'))
        print(f'{p.relative_to(LAKE_DIR)}  ({n} linhas)')

listar_particoes('gold')

gold/alfabetizacao_por_municipio/ano=2023  (5448 linhas)
gold/alfabetizacao_por_municipio/ano=2024  (5448 linhas)
gold/comparacao_metas_nacionais/ano=2024  (1 linhas)
gold/evolucao_temporal/ano=2023  (24 linhas)
gold/evolucao_temporal/ano=2024  (25 linhas)
gold/ranking_municipios/ano=2023  (5448 linhas)
gold/ranking_municipios/ano=2024  (5448 linhas)


In [5]:
snap_exec1 = snapshot_camada(LAKE_DIR, 'gold')
snap_exec1

,particao,n_linhas,hash_negocio
0,gold/alfabetizacao_por_municipio/ano=2023,5448,77962ae3bfe2d20e65c6a2f8a084c756
1,gold/alfabetizacao_por_municipio/ano=2024,5448,07618e3aa32e4d1587fe922beb516f46
2,gold/comparacao_metas_nacionais/ano=2024,1,62f1d1650642e80f4385750d179e2d01
3,gold/evolucao_temporal/ano=2023,24,80c2bcd10b03f3c4bd662c526f0b95c8
4,gold/evolucao_temporal/ano=2024,25,472acddcd7fa23dd27e8e42638abd5c6
5,gold/ranking_municipios/ano=2023,5448,13fdedd5a60a05382cf12cfba7996e9f
6,gold/ranking_municipios/ano=2024,5448,e4d61b197e0f91aa749302335871e5f9


## Execução 2 — reprocesso completo da camada Gold

Repetimos a execução sobre o mesmo insumo, simulando um retry do orquestrador ou um reprocesso manual.

In [6]:
resultado_exec2 = run_gold(LAKE_DIR)
snap_exec2 = snapshot_camada(LAKE_DIR, 'gold')
print('Registros processados na execução 2:')
resultado_exec2

Registros processados na execução 2:


{'alfabetizacao_por_municipio': 10896,
 'evolucao_temporal': 49,
 'ranking_municipios': 10896,
 'comparacao_metas_nacionais': 1}

## Comparação dos snapshots

Partição a partição: mesma estrutura, mesma contagem e mesmo hash de conteúdo de negócio.

In [7]:
comparacao = comparar_snapshots(snap_exec1, snap_exec2)
comparacao

,particao,n_linhas_exec1,hash_negocio_exec1,n_linhas_exec2,hash_negocio_exec2,idempotente
0,gold/alfabetizacao_por_municipio/ano=2023,5448,77962ae3bfe2d20e65c6a2f8a084c756,5448,77962ae3bfe2d20e65c6a2f8a084c756,True
1,gold/alfabetizacao_por_municipio/ano=2024,5448,07618e3aa32e4d1587fe922beb516f46,5448,07618e3aa32e4d1587fe922beb516f46,True
2,gold/comparacao_metas_nacionais/ano=2024,1,62f1d1650642e80f4385750d179e2d01,1,62f1d1650642e80f4385750d179e2d01,True
3,gold/evolucao_temporal/ano=2023,24,80c2bcd10b03f3c4bd662c526f0b95c8,24,80c2bcd10b03f3c4bd662c526f0b95c8,True
4,gold/evolucao_temporal/ano=2024,25,472acddcd7fa23dd27e8e42638abd5c6,25,472acddcd7fa23dd27e8e42638abd5c6,True
5,gold/ranking_municipios/ano=2023,5448,13fdedd5a60a05382cf12cfba7996e9f,5448,13fdedd5a60a05382cf12cfba7996e9f,True
6,gold/ranking_municipios/ano=2024,5448,e4d61b197e0f91aa749302335871e5f9,5448,e4d61b197e0f91aa749302335871e5f9,True


In [8]:
# Validação formal: qualquer divergência interrompe o notebook com erro
assert comparacao['idempotente'].all(), 'FALHA: execuções produziram estados diferentes!'
assert len(snap_exec1) == len(snap_exec2), 'FALHA: número de partições divergente!'

print('=' * 60)
print(f'  IDEMPOTÊNCIA CONFIRMADA — {len(comparacao)} partições idênticas')
print(f'  nas duas execuções (estrutura, contagens e conteúdo).')
print('=' * 60)

  IDEMPOTÊNCIA CONFIRMADA — 7 partições idênticas
  nas duas execuções (estrutura, contagens e conteúdo).


## Amostra das visões analíticas geradas

In [9]:
from pipeline_local import _ler_particionado_por_ano

v1 = _ler_particionado_por_ano(LAKE_DIR / 'gold' / 'alfabetizacao_por_municipio')
print(f'alfabetizacao_por_municipio: {len(v1)} registros')
v1[['id_municipio', 'ano', 'taxa_alfabetizacao', 'meta_2025', 'gap_meta_2025', 'status_meta_2025']].head()

alfabetizacao_por_municipio: 10896 registros


,id_municipio,ano,taxa_alfabetizacao,meta_2025,gap_meta_2025,status_meta_2025
0,1100031,2023,69.10,72.53,-3.43,NAO_ATINGIU
1,1100072,2023,58.20,65.31,-7.11,NAO_ATINGIU
2,1101609,2023,50.70,60.25,-9.55,NAO_ATINGIU
3,1101807,2023,55.69,63.63,-7.94,NAO_ATINGIU
4,1302900,2023,53.17,61.93,-8.76,NAO_ATINGIU


In [10]:
v4 = _ler_particionado_por_ano(LAKE_DIR / 'gold' / 'comparacao_metas_nacionais')
print(f'comparacao_metas_nacionais: {len(v4)} registros')
v4.head()

comparacao_metas_nacionais: 1 registros


,sigla_uf,taxa_uf,meta_nacional_2025,meta_nacional_2030,meta_uf_2025,gap_meta_nacional,gap_meta_uf,status_meta,_gold_processed_at,ano
0,BA,35.96,63.77,80,50.2,-27.81,-14.24,NAO_ATINGIU,20260711_134250,2024


## Conclusão

A camada **Gold** é idempotente: as quatro visões analíticas (`alfabetizacao_por_municipio`, `evolucao_temporal`, `ranking_municipios`, `comparacao_metas_nacionais`) derivam do Silver por operações determinísticas — joins, agregações, window rank e classificações condicionais — e a escrita particionada por ano com overwrite dinâmico garante estado final idêntico a cada execução.

Consequência prática: o consumidor final (Athena, dashboards, modelos de ML) pode confiar que um reprocesso da Gold **nunca** duplica registros nem altera resultados históricos — pré-requisito para usar essas visões como fonte de verdade em decisões de política pública.